# Build a RAG Agent

Completed from the TensorTonic project. The notebook walks through three phases:

1. **Test the LLM on its own** — ask a question whose answer lives only in our document and watch the bare model guess.
2. **Build the retrieval pipeline** — load → chunk → embed → store → retrieve → generate, grounded in the document.
3. **Compare and explore** — ask the same question both ways and probe where RAG holds up.

RAG = retrieval-augmented generation: the model looks information up before it answers instead of answering from memory.

> Note: the code cells read `document.txt` from this directory. Provide that source text to run the pipeline end to end.

## Phase 1 · Ask a bare LLM

Establish a reference point. We ask a question whose answer depends on information the LLM was never trained on (the exact domains TensorTonic's problems cover). With no retrieval the model can only guess or decline — and here it confidently invents domains we don't actually offer.

In [ ]:
import os
from langchain_openai import ChatOpenAI

# Familiar OpenAI interface. The API key and gateway URL come from the environment,
# so this is exactly the code you'd write against OpenAI itself.
llm = ChatOpenAI(model="google.gemma-3-4b-it", base_url=os.environ["OPENAI_BASE_URL"])

# A question whose answer is in the document but the LLM has no way to know
# it on its own. The document lists the exact domains TensorTonic's coding
# problems cover; without retrieval the LLM can only guess or decline.
question = "What domains do TensorTonic's coding problems cover?"
print(llm.invoke([("human", question)]).content)

## Phase 2 · Load the document

The document is the real source of truth. The pipeline forces the LLM to read it before answering: load → chunk → embed/store → retrieve → generate. First, load the raw text into a single LangChain `Document`.

In [ ]:
from langchain_core.documents import Document

text = open("document.txt", encoding="utf-8").read()
doc = Document(page_content=text, metadata={"source": "document.txt"})

print(f"Loaded {len(text):,} characters into 1 document.")

### Concept: chunking and embeddings

A document is too long to feed whole, and we want to search by meaning, not exact words. So:

- **Chunk** — split into passages of a few hundred characters with small overlap so sentences aren't cut in half.
- **Embed** — convert each chunk into a 1024-dim vector encoding its meaning. Similar text lands at nearby "addresses" in meaning space. Vectors are stored in **Chroma**, a local vector database, so we can search by meaning later.

The code cell below still carries the original `TODO` markers for the three steps: build the splitter, split `doc` into chunks, and build the Chroma store.

In [ ]:
import os
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

# TODO 1: create a splitter.
#   - Use RecursiveCharacterTextSplitter.
#   - Pick a chunk size around 512 characters and an overlap around 64.
#   - Docs: https://python.langchain.com/docs/how_to/recursive_text_splitter/
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 512,
    chunk_overlap = 64,
    length_function = len,
    separators = ["\n\n", "\n", ". ", " ", ""],
    is_separator_regex = False,
)  # replace with your splitter

# TODO 2: turn the loaded `doc` into a list of chunks.
#   - Use splitter.split_documents on a list containing your one Document.
chunks = splitter.split_text(text) # replace with your chunks


# The embeddings client. Familiar OpenAI interface, pointed at the gateway.
# Each chunk becomes a 1024-dim vector. (check_embedding_ctx_length=False sends
# raw text, not pre-tokenized ids.)
embeddings = OpenAIEmbeddings(model="amazon.titan-embed-text-v2:0", base_url=os.environ["OPENAI_BASE_URL"], check_embedding_ctx_length=False)

# TODO 3: build a Chroma vector store from your chunks.
#   - Use Chroma.from_documents.
#   - Docs: https://python.langchain.com/docs/integrations/vectorstores/chroma/
vector_store = Chroma.from_texts(
    texts=chunks,
    embedding=embeddings,
    collection_name="my_collection",
)  # replace with your Chroma store

print(f"{len(chunks)} chunks embedded and indexed.")

### Concept: retrieval

Now that every chunk is a vector, searching means: turn the question into a vector too, then find the chunks whose vectors are closest. "Closest" is **cosine distance** in Chroma — lower means more similar. Pull the top `k` chunks (3–5 is a strong default); too many makes the LLM lose focus.

The cell implements `retrieve` (the original `TODO`), then renders the top hits as a text bar chart and projects all chunk vectors into 2D so you can see where the question lands relative to the retrieved chunks.

In [ ]:
import io, base64
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Patches plt.show() to send PNG to the TT frontend. Don't touch.
def _tt_show(*args, **kwargs):
    fig = plt.gcf()
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=80)
    plt.close(fig)
    print(f"__TT_PNG_START__{base64.b64encode(buf.getvalue()).decode()}__TT_PNG_END__")
plt.show = _tt_show

# TODO: implement retrieve.
#   - Inputs: a query string, and k (number of top chunks to return).
#   - Use vector_store.similarity_search_with_score(query, k=k).
#   - Return whatever similarity_search_with_score returns.
#   - Docs: https://python.langchain.com/docs/integrations/vectorstores/chroma/
def retrieve(query: str, k: int = 3):
    return vector_store.similarity_search_with_score(query, k)

hits = retrieve(question)

# Top hits as a simple bar chart in text.
print(f"Top {len(hits)} hits for: {question!r}\n")
max_d = max(s for _, s in hits)
for i, (chunk, score) in enumerate(hits, 1):
    filled = int(40 * (1 - score / max_d))
    bar = "█" * filled + "░" * (40 - filled)
    print(f"[{i}] {bar} {score:.3f}")
    print(f"    {chunk.page_content[:900].replace(chr(10), ' ')}...\n")

# Where chunks live in embedding space, with your question as a red star.
raw = vector_store._collection.get(include=["embeddings", "documents"])
chunk_vecs = np.array(raw["embeddings"])
q_vec = np.array(embeddings.embed_query(question))
hit_texts = {c.page_content for c, _ in hits}
hit_mask = np.array([t in hit_texts for t in raw["documents"]])

joint = np.vstack([chunk_vecs, q_vec])
centered = joint - joint.mean(axis=0)
_, _, Vt = np.linalg.svd(centered, full_matrices=False)
proj = centered @ Vt[:2].T

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(proj[:-1][~hit_mask, 0], proj[:-1][~hit_mask, 1],
           s=60, c="#666", alpha=0.55, label="chunk")
ax.scatter(proj[:-1][hit_mask, 0], proj[:-1][hit_mask, 1],
           s=110, c="#10b981", edgecolor="black", linewidth=0.6, label="retrieved")
ax.scatter(proj[-1, 0], proj[-1, 1],
           s=240, c="#ef4444", marker="*", edgecolor="black", linewidth=0.6, label="question")
ax.set_title("Chunks in embedding space")
ax.set_xlabel("dim 1"); ax.set_ylabel("dim 2")
ax.legend(loc="best", fontsize=9, frameon=False)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

### Concept: grounded prompting

You found relevant chunks — now force the LLM to use **only** them. The request has two jobs:

- **System prompt** sets the rules: answer using only the provided context, cite passages as `[chunk N]`, and say "I don't know" when the context falls short.
- **User message** bundles the retrieved chunks plus the question into one prompt.

The cell defines `SYSTEM` and implements `ask_with_rag` (the original `TODO`): retrieve top-3, label each chunk, and call the model.

In [ ]:
SYSTEM = """You are a question-answering assistant. Answer the user's question using ONLY the information in the provided context passages. Do not use any outside knowledge, and do not make assumptions beyond what the passages state.

Each passage is labeled with a chunk number. When you use information from a passage, cite it inline as [chunk N], where N is that passage's number. Cite every passage you draw from; if a single statement draws on multiple passages, list each one, e.g. [chunk 2][chunk 5].

If the context does not contain enough information to answer the question, do not guess. Respond with: "I don't have enough information in the provided context to answer that." Do not pad the answer with unrelated content from the passages.

Reply in plain text only. Do not use markdown, headings, bullet points, bold, or code formatting of any kind."""

# TODO 2: implement ask_with_rag.
#   - Call retrieve(query, k=3) to get the top hits.
#   - Build a context string: each chunk labelled "[chunk N]" then the chunk text.
#   - Build messages: a system message with SYSTEM, then a human message that
#     includes the context AND the question.
#   - Call llm.invoke(messages) and return the .content string.
#   - Docs: https://python.langchain.com/docs/integrations/chat/bedrock/
def ask_with_rag(query: str) -> str:
    results = retrieve(query, k=3)
    context = "\n\n".join(
        f"[chunk {i}]\n{doc.page_content}"
        for i, (doc, score) in enumerate(results)
    )
    messages = SYSTEM + context
    llm_response = llm.invoke(messages)
    return llm_response.content

print(ask_with_rag(question))

## Phase 3 · Bare LLM vs RAG, side by side

Verify the pipeline actually helps: ask the same question with and without retrieval. Same model, same question — the only difference is whether it had the document to read first.

In [ ]:
def ask_bare(query: str) -> str:
    return llm.invoke([("human", query)]).content

# Same question we asked in cell 1, but this time we run it both ways.
print("BARE LLM (no retrieval)")
print("-" * 40)
print(ask_bare(question))
print()
print("WITH RAG (retrieves from document.txt)")
print("-" * 40)
print(ask_with_rag(question))

### Ask follow-ups

Each question targets a specific detail in the document. The third asks something **not** in the document, so RAG should refuse cleanly instead of inventing an answer.

In [ ]:
# Each question targets a specific detail in the document. The third one
# asks something NOT in the document, so RAG should refuse cleanly.
for q in [
    "What math topics does TensorTonic teach?",
    "What approach does TensorTonic's Research Papers section take?",
    "When was TensorTonic founded?",  # not in doc
]:
    print(f"Q: {q}")
    print(f"A: {ask_with_rag(q)}")
    print()